# Design matrix — habituation sessions (no trial information)

Copy of `paper-individuality/segmentation/2_design_matrix.ipynb`, adapted to habituationChoiceWorld.
Nothing here reads trials: the session window comes from the video itself, and a set of checks
stands in for what the trial times would otherwise guarantee.

**Adaptations to habituation**
- **Left camera only.** Pose and whisker motion energy come from `leftCamera`; licks from
  `get_left_licks` instead of `merge_licks`; no `avg_whisker_me` (needs a second camera). Both paws
  are read from the left camera with the mirrored naming the other one-camera datasets use
  (`l_paw` = `paw_r`, `r_paw` = `paw_l`), as in `1_camera_setup/2.1.1_design_matrix.ipynb`.
- **No wheel.** iblrig-v6 habituation never saved `_iblrig_encoderPositions.raw.ssv`, so there is no
  wheel for any LDA-cohort mouse (see `0_session_inventory.py`).
- **Same 30 Hz grid as the other one-camera datasets.** The native rate (30, 50, 60 or 100 Hz here)
  comes from the median frame interval, and the anti-aliasing low-pass runs only when downsampling.

**Checks replacing the trial times**

1. *Valid camera times* (frames removed). On habituation, `_ibl_leftCamera.times` is only valid while
   the task runs: about 13 s after the last trial every remaining frame is stamped seconds to minutes
   after the previous one (599b970b: 50 s apart, out to 6.4e6 s). The times are split at gaps
   > `MAX_GAP_S` and only segments lasting ≥ `MIN_SEGMENT_S` are kept, which drops that tail and any
   stray frames. A genuine recording gap inside the session keeps both sides, and the bins in the
   gap are NaN (`mask_gaps`), never interpolated.
2. *Untracked edges* (frames removed). Leading and trailing stretches where the mouse's left paw is
   tracked in < `EDGE_MIN_TRACKED` of a `EDGE_WINDOW_S` window are cut, which removes handling at
   either end.
3. *Per-signal usability* (flags only, nothing removed). The camera placement differs between rigs
   and days, and in some sessions the nose, tube and tongue are out of frame (nose at x ≈ 2 px in
   3dc44d45) while the paws are fine. Each signal gets its own flag in `processing_log.csv`:
   - `paw_ok`: left paw (the paw used for segmentation) NaN in ≤ `MAX_PAW_NAN` of bins
   - `lick_ok`: nose and spout tracked in ≥ `MIN_FACE_TRACKED` of frames (otherwise licks cannot
     be detected, and a lick count of zero would look like a mouse that does not lick)
   - `whisk_ok`: nose tracked in ≥ `MIN_FACE_TRACKED` of frames (the whisker-pad ROI is placed
     relative to the face) and motion energy is not constant
   - `duration_ok`: ≥ `MIN_DURATION_MIN` minutes left after checks 1–2
4. *Validation* (last section). The trials of every session, saved by `trial_window_reference.py` in
   `trial_window_reference/`, give the task window the windows from checks 1–2 are compared with.

Writes `design_matrix_<eid>_<mouse>` and `processing_log.csv` to `data/individuality/habituation/`.

In [ ]:
import os
import sys
import gc

import numpy as np
import pandas as pd
from brainbox.io.one import SessionLoader
from one.api import ONE

prefix = '/home/ines/repositories/'
sys.path.insert(0, prefix + 'representation_learning_variability/paper-individuality/segmentation/')
from segmentation_functions import get_left_licks, resample_common_time, interpolate_nans, low_pass

one = ONE(mode='remote')

In [ ]:
FS = 30                 # output grid (Hz), same as the other one-camera datasets
LOW_PASS_CUTOFF = 12    # Hz, anti-aliasing before downsampling to FS (as in 2.1.1)
LICK_FS = 60            # grid of the lick event trace before resampling to FS (as in 2.1.1)

# Check 1: valid camera times
MAX_GAP_S = 1           # frame intervals above this split the recording into segments
MIN_SEGMENT_S = 60      # segments shorter than this are dropped (the broken tail is 1-frame segments)
# Check 2: untracked edges
EDGE_WINDOW_S = 10
EDGE_MIN_TRACKED = 0.5
# Check 3: per-signal usability flags
MAX_PAW_NAN = 0.2
MIN_FACE_TRACKED = 0.5
MIN_DURATION_MIN = 10

data_path = prefix + 'representation_learning_variability/data/individuality/habituation/'
inventory = pd.read_csv(prefix + 'representation_learning_variability/habituation/habituation_lp_sessions.csv')

### Sessions

`habituation_lp_sessions.csv` (from `0_session_inventory.py`): every habituation session with a
leftCamera LightningPose file. Camera times and whisker motion energy are needed.

In [ ]:
usable = inventory[inventory['lp_left'] & inventory['cam_times_left'] & inventory['me_left']]
print(f'{len(usable)} of {len(inventory)} sessions have pose, camera times and motion energy')

sessions_to_process = [e for e, m in zip(usable['eid'], usable['mouse_name'])
                       if not os.path.exists(data_path + f'design_matrix_{e}_{m}')]
print(f'{len(sessions_to_process)} sessions to process')

In [ ]:
def valid_time_frames(t, max_gap=MAX_GAP_S, min_segment=MIN_SEGMENT_S):
    """ Check 1. Boolean mask of frames in segments (split at intervals > max_gap) that last at
    least min_segment seconds. Drops the run of broken timestamps habituation videos have after
    the task, and stray trailing frames. """

    breaks = np.where(np.diff(t) > max_gap)[0] + 1
    keep = np.zeros(len(t), dtype=bool)
    for seg in np.split(np.arange(len(t)), breaks):
        if len(seg) and t[seg[-1]] - t[seg[0]] >= min_segment:
            keep[seg] = True
    return keep


def tracked_edges(tracked, fr, window_s=EDGE_WINDOW_S, min_tracked=EDGE_MIN_TRACKED):
    """ Check 2. Boolean mask that cuts leading and trailing stretches where the fraction of
    tracked frames in a centred window of window_s seconds is below min_tracked. """

    w = max(1, int(round(window_s * fr)))
    frac = pd.Series(tracked.astype(float)).rolling(w, center=True, min_periods=1).mean().to_numpy()
    ok = np.where(frac >= min_tracked)[0]
    keep = np.zeros(len(tracked), dtype=bool)
    if len(ok):
        keep[ok[0]:ok[-1] + 1] = True
    return keep


def mask_gaps(values, ref_t, t, max_gap=0.1):
    """ resample_common_time interpolates linearly across missing timestamps, which fabricates
    data over stretches where the camera was not recording (e.g. 6f09ba7e has 89 s of gap, which
    would otherwise become a smooth ramp of paw position and motion energy). Set those bins to
    NaN instead, using the same 0.1 s threshold interpolate_nans uses for runs of NaNs. """

    masked = np.asarray(values, dtype=np.float64).copy()
    for g in np.where(np.diff(t) > max_gap)[0]:
        masked[(ref_t > t[g]) & (ref_t < t[g + 1])] = np.nan
    return masked


def process_design_matrix(session):

    mouse_name = inventory.set_index('eid').loc[session, 'mouse_name']
    log = {'eid': session, 'mouse_name': mouse_name}

    """ LOAD VARIABLES """
    sl = SessionLoader(eid=session, one=one)
    # likelihood < 0.9 -> NaN (SessionLoader default)
    sl.load_pose(views=['left'], tracker='lightningPose')
    # Left camera only: load_session_data's default views include right/body, which raise on
    # one-camera sessions and can leave sl.motion_energy without 'leftCamera' (see 2.1.1)
    sl.load_motion_energy(views=['left'])
    pose = sl.pose['leftCamera']
    me = sl.motion_energy['leftCamera']
    log['n_frames'] = len(pose)

    # Check 1: valid camera times (pose and motion energy are frame-aligned)
    keep = valid_time_frames(np.asarray(pose['times']))
    log['n_frames_bad_times'] = int((~keep).sum())
    if not keep.any():
        log['status'] = 'no valid camera times'
        return log
    pose = pose.loc[keep].reset_index(drop=True)
    me = me.loc[keep].reset_index(drop=True)

    # Nominal rate from the median interval, robust to dropped frames (not get_frame_rate: it
    # snaps to 30/60/150 Hz and some habituation videos are 50 or 100 Hz)
    left_fr = int(round(1 / np.median(np.diff(np.asarray(pose['times'])))))
    log['left_fr'] = left_fr
    if left_fr < FS:
        log['status'] = 'frame rate too low'
        return log

    # Check 2: untracked edges (left paw = the camera's paw_r)
    keep = tracked_edges(pose['paw_r_x'].notna().to_numpy(), left_fr)
    log['n_frames_untracked_edges'] = int((~keep).sum())
    if not keep.any():
        log['status'] = 'left paw never tracked'
        return log
    pose = pose.loc[keep].reset_index(drop=True)
    me = me.loc[keep].reset_index(drop=True)

    lc_t = np.asarray(pose['times'])
    poses = {'leftCamera': pose}       # get_left_licks takes the SessionLoader layout

    def prepare(x):
        # Anti-alias only when downsampling to the FS grid
        x = interpolate_nans(x, left_fr)
        return low_pass(x, cutoff=LOW_PASS_CUTOFF, sf=left_fr) if left_fr > FS else x

    # Motion energy
    motion_energy_l = prepare(me['whiskerMotionEnergy'])
    # Licks
    features = ['tongue_end_l_x', 'tongue_end_l_y', 'tongue_end_r_x', 'tongue_end_r_y']
    lick_t, licks = get_left_licks(poses, features, common_fs=LICK_FS)
    # Paws (a single camera sees both paws; the left camera's paw_r is the mouse's left paw)
    l_paw_x = prepare(pose['paw_r_x'])
    l_paw_y = prepare(pose['paw_r_y'])
    r_paw_x = prepare(pose['paw_l_x'])
    r_paw_y = prepare(pose['paw_l_y'])

    # Common resampling window
    onset = max(lc_t.min(), lick_t.min())
    offset = min(lc_t.max(), lick_t.max())
    ref_t = np.arange(onset, offset, 1 / FS, dtype=np.float64)

    # Restrict to time window
    def restrict(t, x):
        mask = (t >= onset) & (t <= offset)
        return t[mask], np.asarray(x)[mask]

    cam_t, motion_energy_l = restrict(lc_t, motion_energy_l)
    _, l_paw_x = restrict(lc_t, l_paw_x)
    _, l_paw_y = restrict(lc_t, l_paw_y)
    _, r_paw_x = restrict(lc_t, r_paw_x)
    _, r_paw_y = restrict(lc_t, r_paw_y)
    lick_t, licks = restrict(lick_t, licks)

    # Resample
    me_down, rt = resample_common_time(ref_t, cam_t, motion_energy_l, kind='linear')
    lk_down, _ = resample_common_time(ref_t, lick_t, licks, kind='nearest')
    lpx_down, _ = resample_common_time(ref_t, cam_t, l_paw_x, kind='linear')
    lpy_down, _ = resample_common_time(ref_t, cam_t, l_paw_y, kind='linear')
    rpx_down, _ = resample_common_time(ref_t, cam_t, r_paw_x, kind='linear')
    rpy_down, _ = resample_common_time(ref_t, cam_t, r_paw_y, kind='linear')

    # Blank out the bins the camera did not record, rather than interpolating across them.
    # Lick count is left alone (it stays int8); the camera columns already flag those bins,
    # so dropping NaN bins downstream drops them too.
    me_down = mask_gaps(me_down, ref_t, cam_t)
    lpx_down = mask_gaps(lpx_down, ref_t, cam_t)
    lpy_down = mask_gaps(lpy_down, ref_t, cam_t)
    rpx_down = mask_gaps(rpx_down, ref_t, cam_t)
    rpy_down = mask_gaps(rpy_down, ref_t, cam_t)

    # Create design matrix
    design_matrix = pd.DataFrame({
        'Bin': rt,
        'Lick count': lk_down.astype(np.int8),
        'whisker_me': me_down,
        'l_paw_x': lpx_down,
        'l_paw_y': lpy_down,
        'r_paw_x': rpx_down,
        'r_paw_y': rpy_down,
    })

    # Check 3: per-signal usability flags
    nose_tracked = pose['nose_tip_x'].notna().mean()
    spout_tracked = pose[['tube_top_x', 'tube_bottom_x']].notna().any(axis=1).mean()
    duration_min = (rt[-1] - rt[0]) / 60
    log.update({
        'status': 'ok',
        'window_start': rt[0], 'window_end': rt[-1], 'duration_min': duration_min,
        'n_bins': len(design_matrix),
        'l_paw_nan': design_matrix['l_paw_x'].isna().mean(),
        'nose_tracked': nose_tracked, 'spout_tracked': spout_tracked,
        'lick_bins_per_min': design_matrix['Lick count'].sum() / duration_min,
        'whisker_me_std': np.nanstd(me_down),
    })
    log['paw_ok'] = log['l_paw_nan'] <= MAX_PAW_NAN
    log['lick_ok'] = (nose_tracked >= MIN_FACE_TRACKED) and (spout_tracked >= MIN_FACE_TRACKED)
    log['whisk_ok'] = (nose_tracked >= MIN_FACE_TRACKED) and (log['whisker_me_std'] > 0)
    log['duration_ok'] = duration_min >= MIN_DURATION_MIN

    """ SAVE DATA """
    # Save unnormalized design matrix
    filename = data_path + "design_matrix_" + str(session) + '_' + mouse_name
    design_matrix.to_parquet(filename, compression='gzip')

    del design_matrix, sl, pose, me
    gc.collect()
    return log

### Run

One session at a time (LP files are 30-80 MB each). Failures are logged rather than stopping the
loop; `processing_log.csv` is appended to on every run.

In [ ]:
logs = []
for s, session in enumerate(sessions_to_process):
    try:
        logs.append(process_design_matrix(session))
    except Exception as e:
        print(f'Failed session {session}: {e!r}')
        logs.append({'eid': session, 'status': f'failed: {e!r}'})
    print(f'[{s + 1}/{len(sessions_to_process)}] {session}: {logs[-1]["status"]}', flush=True)

log_file = data_path + 'processing_log.csv'
log_df = pd.DataFrame(logs)
if os.path.exists(log_file) and len(log_df):
    log_df = pd.concat([pd.read_csv(log_file), log_df]).drop_duplicates('eid', keep='last')
elif os.path.exists(log_file):
    log_df = pd.read_csv(log_file)
log_df.to_csv(log_file, index=False)
log_df['status'].value_counts()

### Summary of the checks

In [ ]:
ok = log_df[log_df['status'] == 'ok']
flags = ['paw_ok', 'lick_ok', 'whisk_ok', 'duration_ok']
print(f'{len(ok)} sessions processed; frame rates: {ok["left_fr"].value_counts().to_dict()}')
print(f'frames removed: bad times {ok["n_frames_bad_times"].sum() / ok["n_frames"].sum():.1%}, '
      f'untracked edges {ok["n_frames_untracked_edges"].sum() / ok["n_frames"].sum():.1%}')
print(ok[flags].mean().round(3).to_string())
ok.loc[~ok[flags].all(axis=1), ['eid', 'mouse_name'] + flags + ['l_paw_nan', 'nose_tracked', 'spout_tracked', 'duration_min']]

### Validation against the task window

`trial_window_reference/` (written by `trial_window_reference.py`) holds the trials of every session (ALF, or extracted from the raw task data
where ALF trials were missing). The window found by checks 1–2 should start at the first trial and
end within seconds of the last one. Not part of the pipeline; skipped if the folder is absent.

In [ ]:
ref_path = data_path + 'trial_window_reference/'
if os.path.isdir(ref_path):
    rows = []
    for _, r in ok.iterrows():
        f = ref_path + f'session_trials_{r.eid}_{r.mouse_name}'
        if os.path.exists(f):
            tr = pd.read_parquet(f)
            rows.append({'eid': r.eid, 'mouse_name': r.mouse_name,
                         'start_diff_s': r.window_start - tr['goCueTrigger_times'].iloc[0],
                         'end_diff_s': r.window_end - tr['intervals_1'].iloc[-1]})
    validation = pd.DataFrame(rows)
    print(validation[['start_diff_s', 'end_diff_s']].describe().round(1).to_string())
    print('\nsessions whose window differs from the task by more than 60 s at either end:')
    print(validation.loc[(validation[['start_diff_s', 'end_diff_s']].abs() > 60).any(axis=1)].round(1).to_string())